## For each dyad, we want to know if the participants' EEG signals are more phase-synchronized during their actual joint trials than during mismatched trials

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import mne
from scipy.signal import hilbert

Load locations

In [6]:
config = json.loads(Path("data/config.json").read_text())

DATASET_PATH = Path(config["dataset_path"]).expanduser()
DERIVATIVES_PATH = Path(config["derivatives_path"]).expanduser()

EPOCHS_DIR = DERIVATIVES_PATH / "epochs"
SUMMARY_PATH = Path("preprocessing_summary.csv")

print("Dataset:", DATASET_PATH)
print("Epochs:", EPOCHS_DIR)
print("Summary exists:", SUMMARY_PATH.exists())

Dataset: /Volumes/untitled/ds007471
Epochs: /Volumes/Untitled/ds007471_derivatives/epochs
Summary exists: True


Get preprocessing results

In [7]:
summary = pd.read_csv(SUMMARY_PATH)

print(summary.shape)
print(summary.columns.tolist())
summary.head()

(32, 7)
['sub', 'boundary', 'practice', 'real', 'complete', 'duet', 'constant_pitch']


,sub,boundary,practice,real,complete,duet,constant_pitch
0,sub-01,S1,6,57,39,19,20
1,sub-02,S1,6,58,37,19,18
2,sub-03,S107 fallback,6,56,40,20,20
3,sub-04,S1,6,57,37,19,18
4,sub-05,S1,6,56,39,20,19


In [ ]:
plv_results = {}

for subname in summary["sub"]:

    epochs_R = mne.read_epochs(
        EPOCHS_DIR / f"{subname}_R_long-epo.fif",
        preload=True
    )

    epochs_L = mne.read_epochs(
        EPOCHS_DIR / f"{subname}_L_long-epo.fif",
        preload=True
    )

    # Bandpass filter (theta)
    theta_R = epochs_R.copy().filter(
        l_freq=4,
        h_freq=8,
        picks="eeg"
    )

    theta_L = epochs_L.copy().filter(
        l_freq=4,
        h_freq=8,
        picks="eeg"
    )

    # Extract analytic signal
    theta_R.apply_hilbert(envelope=False)
    theta_L.apply_hilbert(envelope=False)

    # Crop to time window of interest
    theta_R.crop(tmin=4.5, tmax=12.5)
    theta_L.crop(tmin=4.5, tmax=12.5)

    # Extract instantaneous phase
    phase_R = np.angle(theta_R.get_data())
    phase_L = np.angle(theta_L.get_data())

    n_trials = phase_R.shape[0]
    T = phase_R.shape[-1]

    plv_values = []

    for i in range(n_trials):

        trial_R = phase_R[i]
        trial_L = phase_L[i]

        # Transform into complex unit vectors
        U_R = np.exp(1j * trial_R)
        U_L = np.exp(1j * trial_L)

        # Calculate cross-phase between all channel pairs
        cross_phase = U_R @ U_L.conj().T

        # 27x27 PLV matrix
        plv_matrix = np.abs(cross_phase) / T

        # Mean PLV per trial
        trial_plv = plv_matrix.mean()
        plv_values.append(trial_plv)

    # Store results for this subject
    plv_results[subname] = plv_values

    print(f"{subname}: {len(plv_values)} trials, mean PLV = {np.mean(plv_values):.4f}")
    
plv_df = pd.DataFrame({
    sub: pd.Series(values)
    for sub, values in plv_results.items()
})

plv_df.to_csv("plv_results.csv", index_label="trial")